# Example: Multi-Trajectory Ensemble Averaging

Demonstrates how multiple independent trajectories improve spectral
estimation by ensemble averaging. The same system is measured L times
with different inputs and noise -- averaging reduces variance by factor L
without sacrificing frequency resolution.

Functions demonstrated: `sid.freq_bt`, `sid.freq_map`, `sid.spectrogram`, `sid.ltv_disc`, `sid.ltv_disc_frozen`.

In [ ]:
import numpy as np
from scipy.signal import lfilter
import matplotlib.pyplot as plt
import sid

%matplotlib inline

## 1. LTI Ensemble Averaging -- Tighter Confidence Bands

Generate L=10 trajectories of a known second-order system.
Compare `sid.freq_bt` with 1 vs 10 trajectories.

In [ ]:
rng = np.random.default_rng(5001)
N, L, Ts = 2000, 10, 1.0
b_true = [1]
a_true = [1, -2 * 0.9 * np.cos(np.pi / 4), 0.9**2]

y_all = np.zeros((N, 1, L))
u_all = np.zeros((N, 1, L))
for l in range(L):
    u_all[:, 0, l] = rng.standard_normal(N)
    y_all[:, 0, l] = lfilter(b_true, a_true, u_all[:, 0, l]) + 0.3 * rng.standard_normal(N)

# Single trajectory
r1 = sid.freq_bt(y_all[:, :, 0], u_all[:, :, 0], window_size=40)

# Multi-trajectory ensemble (all L=10)
rL = sid.freq_bt(y_all, u_all, window_size=40)

print(f'Single trajectory: max response_std = {r1.response_std.max():.4f}')
print(f'10 trajectories:   max response_std = {rL.response_std.max():.4f}')
print(f'Ratio: {rL.response_std.max() / r1.response_std.max():.2f} '
      f'(expected ~{1 / np.sqrt(L):.2f} = 1/sqrt({L}))')

# Plot comparison
fig, axes = plt.subplots(2, 1, figsize=(8, 8))
sid.bode_plot(r1, confidence=3, ax=(axes[0], axes[0].twinx()))
axes[0].set_title('Single Trajectory')
sid.bode_plot(rL, confidence=3, ax=(axes[1], axes[1].twinx()))
axes[1].set_title(f'Ensemble of {L} Trajectories')
plt.tight_layout()
plt.show()

## 2. LTV Time-Varying Map -- Sharper Transition Detection

System with a step change at `t = N/2`: pole shifts from 0.85 to 0.5.

In [ ]:
rng = np.random.default_rng(5002)
N, L = 4000, 5
y_ltv = np.zeros((N, 1, L))
u_ltv = np.zeros((N, 1, L))
for l in range(L):
    ul = rng.standard_normal(N)
    u_ltv[:, 0, l] = ul
    yl = np.zeros(N)
    for t in range(1, N):
        a_t = 0.85 if t <= N // 2 else 0.5
        yl[t] = a_t * yl[t - 1] + ul[t] + 0.2 * rng.standard_normal()
    y_ltv[:, 0, l] = yl

# Single trajectory map
r1_map = sid.freq_map(y_ltv[:, :, 0], u_ltv[:, :, 0], segment_length=500)

# Multi-trajectory map
rL_map = sid.freq_map(y_ltv, u_ltv, segment_length=500)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sid.map_plot(r1_map, plot_type='magnitude', ax=axes[0])
axes[0].set_title('Single Trajectory')
sid.map_plot(rL_map, plot_type='magnitude', ax=axes[1])
axes[1].set_title(f'Ensemble of {L} Trajectories')
plt.tight_layout()
plt.show()

## 3. Spectrogram Averaging -- Chirp in Noise

A chirp signal buried in noise. Single trajectory: barely visible.
8 trajectories averaged: chirp track emerges clearly.

In [ ]:
rng = np.random.default_rng(5003)
N, L, Fs = 2000, 8, 1000
Ts = 1 / Fs
t = np.arange(N) / Fs
f0, f1 = 50, 400
chirp_sig = np.sin(2 * np.pi * (f0 * t + (f1 - f0) / (2 * t[-1]) * t**2))

x_all = np.zeros((N, 1, L))
for l in range(L):
    x_all[:, 0, l] = 0.3 * chirp_sig + rng.standard_normal(N)

r1_spec = sid.spectrogram(x_all[:, :, 0], window_length=128, sample_time=Ts)
rL_spec = sid.spectrogram(x_all, window_length=128, sample_time=Ts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sid.spectrogram_plot(r1_spec, ax=axes[0])
axes[0].set_title('Single Trajectory')
sid.spectrogram_plot(rL_spec, ax=axes[1])
axes[1].set_title(f'Ensemble of {L} Trajectories')
plt.tight_layout()
plt.show()

## 4. COSMIC + `sid.freq_map` Consistency

Use the same multi-trajectory dataset for both COSMIC (state-space)
and `sid.freq_map` (non-parametric frequency). Compare frozen transfer
function from COSMIC against the ensemble-averaged spectral map.

In [ ]:
rng = np.random.default_rng(5004)
p, q, N, L_traj = 2, 1, 50, 10
A_true = np.array([[0.9, 0.1], [-0.05, 0.8]])
B_true = np.array([[0.5], [0.3]])
sigma = 0.02

X = np.zeros((N + 1, p, L_traj))
U = rng.standard_normal((N, q, L_traj))
for l in range(L_traj):
    X[0, :, l] = 0.1 * rng.standard_normal(p)
    for k in range(N):
        X[k + 1, :, l] = (A_true @ X[k, :, l]
                          + B_true @ U[k, :, l]
                          + sigma * rng.standard_normal(p))

# COSMIC identification
ltv = sid.ltv_disc(X, U, lambda_=1e4, uncertainty=True)
print(f'COSMIC identified A(0) =\n{ltv.a[:, :, 0]}')

# Frozen transfer function
frz = sid.ltv_disc_frozen(ltv)

# Ensemble-averaged sid.freq_map (treating state x1 as output, u as input)
y_freq = X[:N, 0:1, :].copy()  # shape (N, 1, L_traj)
u_freq = U.copy()               # shape (N, 1, L_traj)
fmap = sid.freq_map(y_freq, u_freq, segment_length=min(N, 30))

print(f'sid.freq_map ensemble with {fmap.num_trajectories} trajectories: '
      f'{len(fmap.time)} segments.')
print(f'Both COSMIC and sid.freq_map use the same {L_traj}-trajectory dataset.')